In [1]:
import cv2
import time
import math
import mediapipe as mp


from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision.hand_landmarker import (
    HandLandmarker,
    HandLandmarkerOptions,
    HandLandmarksConnections
)

In [2]:
print(dir(mp))

['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [3]:
BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode
HandLandmarkerResult = mp.tasks.vision.HandLandmarkerResult


options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="hand_landmarker.task"),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=2
    )

landmarker = HandLandmarker.create_from_options(options)
HAND_CONNECTIONS = HandLandmarksConnections.HAND_CONNECTIONS

In [ ]:
""" Function to  calculates FPS"""
def calcFPS(curr_time,prev_time):
    fps = 1/(curr_time - prev_time)
    return fps

In [5]:
HAND_CONNECTIONS

[HandLandmarksConnections.Connection(start=0, end=1),
 HandLandmarksConnections.Connection(start=1, end=5),
 HandLandmarksConnections.Connection(start=9, end=13),
 HandLandmarksConnections.Connection(start=13, end=17),
 HandLandmarksConnections.Connection(start=5, end=9),
 HandLandmarksConnections.Connection(start=0, end=17),
 HandLandmarksConnections.Connection(start=1, end=2),
 HandLandmarksConnections.Connection(start=2, end=3),
 HandLandmarksConnections.Connection(start=3, end=4),
 HandLandmarksConnections.Connection(start=5, end=6),
 HandLandmarksConnections.Connection(start=6, end=7),
 HandLandmarksConnections.Connection(start=7, end=8),
 HandLandmarksConnections.Connection(start=9, end=10),
 HandLandmarksConnections.Connection(start=10, end=11),
 HandLandmarksConnections.Connection(start=11, end=12),
 HandLandmarksConnections.Connection(start=13, end=14),
 HandLandmarksConnections.Connection(start=14, end=15),
 HandLandmarksConnections.Connection(start=15, end=16),
 HandLandmark

In [ ]:
cap = cv2.VideoCapture(0)

prev_time = 0

Index_tip = 8
Thumb_tip = 4
Middle_Finger_MCP = 9

# for tracking and drawing index_finger_tip points>>>
draw_points = []
prev_index_pt= None
Draw_Threshold = 100

while True:
    ret,img = cap.read()
    img = cv2.flip(img, 1)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb)

    """ showing FPS meter on screen """
    curr_time = time.time()
    fps = calcFPS(curr_time,prev_time)
    prev_time = curr_time
    cv2.putText(img , f"FPS: {fps:.2f}",(50,50),cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),2)
    
    timestamp = int(time.time() * 1000)
    result = landmarker.detect_for_video(mp_image , timestamp)
    timestamp += 1

    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            h, w, _ = img.shape
            points = []

            
            # Convert normalized landmarks to pixel coordinates
            for lm in hand_landmarks:
                x, y = int(lm.x * w), int(lm.y * h)
                # print(x,y)
                
                points.append((x, y))
                cv2.circle(img, (x, y), 4, (0, 255, 0), -1)
            
                
            
            # Draw connections between landmarks
            for conn in HAND_CONNECTIONS:
                start = conn.start
                end = conn.end
                cv2.line(img, points[start], points[end], (255, 0, 0), 3)

                #Draw line between 8 and 4
                # Thumb_tip = 4
                # Index_tip = 8

                #calculate and show dist of 8 and 4
                
                # dist = int(math.dist(points[Thumb_tip] , points[Index_tip]))
                # if (dist-15)<101:
                #     cv2.line(img, points[Thumb_tip], points[Index_tip], (255, 0, 0), 3)
                #     cv2.putText(img , f"Dist : {dist-15}",(50,100),cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),2)

            index_pt = points[Index_tip]
            thumb_pt = points[Thumb_tip]

            distance = math.dist(index_pt, thumb_pt)

            if distance<Draw_Threshold:
                if prev_index_pt is None:
                    prev_index_pt = index_pt
                draw_points.append((prev_index_pt, index_pt))
                prev_index_pt = index_pt
            else:
                prev_index_pt = None


            # Draw a box/rectangle
            xcenter_pts = [p[0] for p in points]
            ycenter_pts = [p[1] for p in points]
            
            xmin = min(xcenter_pts)
            ymin = min(ycenter_pts)
            xmax = max(xcenter_pts)
            ymax = max(ycenter_pts)
            cv2.rectangle(img,(xmin-100,ymin-100),(ymin+100,ymax+100),(255,0,0),3)
            
    for p1, p2 in draw_points:
        cv2.line(img, p1, p2, (0, 0, 255), 4)
    
    cv2.imshow("Image", img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
                
cap.release()
cv2.destroyAllWindows()

error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1301: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvShowImage'
